In [0]:
--  set catalog + schema
USE CATALOG ymotorna_DB;
USE SCHEMA superstore_lakehouse;

-- check
SELECT current_catalog(), 
    current_schema(); 

current_catalog(),current_schema()
ymotorna_db,superstore_lakehouse


In [0]:
show tables;

database,tableName,isTemporary
superstore_lakehouse,bronze_orders,false
superstore_lakehouse,gold_customer_metrics,false
superstore_lakehouse,gold_sales_category,false
superstore_lakehouse,gold_sales_daily,false
superstore_lakehouse,gold_sales_region,false
superstore_lakehouse,products,false
superstore_lakehouse,silver_orders,false
superstore_lakehouse,silver_rejected_orders,false


### Create gold layer

In [0]:
-- drop table if exists gold_customer_metrics;
-- drop table if exists gold_sales_category;
-- drop table if exists gold_sales_daily;
-- drop table if exists gold_sales_region;
-- drop table if exists gold_products;

-- gold_sales_daily
-- | date | revenue |
create table if not exists gold_sales_daily (
    date date,
    revenue double,
    profit double
);


-- gold_sales_region
-- | region | revenue |
create table if not exists gold_sales_region (
    region string,
    revenue double
);




-- gold_sales_category
-- | category | revenue |
create table if not exists gold_sales_category (
    category string,
    revenue double
);



-- gold_customer_metrics
-- | customer | revenue | orders |
create table if not exists gold_customer_metrics (
    customer string,
    revenue double,
    orders long
);



-- products
create table if not exists gold_products (
    product_id string,
    product_name string,
    category string,
    sub_category string,
    region string,
    total_revenue double,
    quantity_sold long
);



show tables;

database,tableName,isTemporary
superstore_lakehouse,bronze_orders,false
superstore_lakehouse,gold_customer_metrics,false
superstore_lakehouse,gold_products,false
superstore_lakehouse,gold_sales_category,false
superstore_lakehouse,gold_sales_daily,false
superstore_lakehouse,gold_sales_region,false
superstore_lakehouse,products,false
superstore_lakehouse,silver_orders,false
superstore_lakehouse,silver_rejected_orders,false


Insert data


In [0]:
-- gold_sales_daily
-- | date | revenue |
insert overwrite gold_sales_daily
select order_date as date,
    sum(sales) as revenue,
    sum(profit) as profit
from silver_orders
group by date 
order by date desc;


-- gold_sales_region
-- | region | revenue |
insert overwrite gold_sales_region
select region,
    sum(sales) as revenue
from silver_orders
group by region
order by revenue desc;



-- gold_sales_category
-- | category | revenue |
insert overwrite gold_sales_category
select category,
    sum(sales) as revenue
from silver_orders
group by category
order by revenue desc;


-- gold_customer_metrics
-- | customer | revenue | orders |
insert overwrite gold_customer_metrics
select customer_name as customer,
    sum(sales) as revenue,
    count(distinct order_id) as orders
from silver_orders
group by customer
order by revenue desc;


-- products
insert overwrite gold_products 
select
    product_id,
    product_name,
    category,
    sub_category,
    region,
    sum(sales) as total_revenue,
    sum(quantity) as quantity_sold
from silver_orders
group by 1,2,3,4,5;


show tables;

database,tableName,isTemporary
superstore_lakehouse,bronze_orders,false
superstore_lakehouse,gold_customer_metrics,false
superstore_lakehouse,gold_products,false
superstore_lakehouse,gold_sales_category,false
superstore_lakehouse,gold_sales_daily,false
superstore_lakehouse,gold_sales_region,false
superstore_lakehouse,products,false
superstore_lakehouse,silver_orders,false
superstore_lakehouse,silver_rejected_orders,false


Check


In [0]:
select 'gold_sales_daily' as tbl, 
    count(*) as records from gold_sales_daily
union all
select 'gold_sales_region' as tbl, 
    count(*) as records from gold_sales_region
union all
select 'gold_sales_category' as tbl, 
    count(*) as records from gold_sales_category
union all
select 'gold_customer_metrics' as tbl, 
    count(*) as records from gold_customer_metrics
union all
select 'gold_products' as tbl, 
    count(*) as records from gold_products;


tbl,records
gold_sales_daily,1234
gold_sales_region,4
gold_sales_category,3
gold_customer_metrics,795
gold_products,5220


In [0]:
select count(distinct order_date) as dates,
    count(distinct region) as regions,
    count(distinct category) as categories,
    count(distinct customer_name) as customers
from silver_orders;

dates,regions,categories,customers
1234,4,3,795
